In [9]:
import torch
import numpy as np
from tqdm import tqdm
import os
from torchvision import transforms
from safetensors.torch import load_file

# ===================== 配置 =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224
NPZ_DIR = "/autodl-tmp/figure_separate"  # 也同步改成/autodl-tmp/
SAVE_FEATURE_DIR = "/autodl-tmp/f_s_mae_features"
LOCAL_MAE_PATH = "/autodl-tmp/new_mae_model"  # ✅ 修复后的正确路径

os.makedirs(SAVE_FEATURE_DIR, exist_ok=True)

# ===================== 【离线加载】真正 MAE 模型 =====================
print("🚀 正在从本地加载 MAE 模型...")

import timm
# 1. 创建真正的 MAE 结构
model = timm.create_model(
    "vit_base_patch16_224.mae",
    pretrained=False,
    img_size=224
)

# 2. 加载你本地权重
state_dict = load_file(os.path.join(LOCAL_MAE_PATH, "model.safetensors"))
model.load_state_dict(state_dict, strict=False)  # 跳过不匹配层

# 3. 取 encoder 做特征提取
encoder = model.encoder.to(DEVICE)
encoder.eval()

print("✅ MAE 模型本地加载成功！")

# ===================== 预处理 =====================
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ===================== 特征提取 =====================
def extract_feature(img_np):
    with torch.no_grad():
        img = preprocess(img_np).unsqueeze(0).to(DEVICE)
        feat = encoder(img)[:, 0, :].squeeze(0).cpu().numpy()
    return feat

# ===================== 批量提取 =====================
def extract_all():
    files = [f for f in os.listdir(NPZ_DIR) if f.endswith(".npz")]
    print(f"📊 共 {len(files)} 只股票")

    for f in tqdm(files, desc="MAE 特征提取"):
        sid = f[:-4]
        data = np.load(os.path.join(NPZ_DIR, f))
        
        imgs = data["cv"]
        labels = data["label"]
        
        feats = [extract_feature(img) for img in imgs]
        np.savez_compressed(
            os.path.join(SAVE_FEATURE_DIR, f"{sid}_mae.npz"),
            feature=np.array(feats),
            label=labels
        )

    print("🎉 全部完成！图像特征已准备好用于多模态训练！")

if __name__ == "__main__":
    extract_all()

🚀 正在从本地加载 MAE 模型...


FileNotFoundError: No such file or directory: /autodl-tmp/new_mae_model/model.safetensors

In [3]:
import torch
import numpy as np
from tqdm import tqdm
import os
from torchvision import transforms
from safetensors.torch import load_file

# ===================== 配置 =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224

# ✅ AutoDL 真实路径（全部正确）
NPZ_DIR = "/root/autodl-tmp/figure_separate"
SAVE_FEATURE_DIR = "/root/autodl-tmp/f_s_vit_features"
LOCAL_MODEL_FILE = "/root/autodl-tmp/model.safetensors"  # ✅ 正确路径

os.makedirs(SAVE_FEATURE_DIR, exist_ok=True)

# ===================== 加载本地模型 =====================
print("🚀 加载本地视觉模型...")

import timm
# 创建模型（特征提取模式）
model = timm.create_model(
    "vit_base_patch16_224", 
    pretrained=False, 
    img_size=224,
    num_classes=0  # 直接输出768维特征
)

# 加载本地权重
state_dict = load_file(LOCAL_MODEL_FILE)
model.load_state_dict(state_dict, strict=False)
model = model.to(DEVICE)
model.eval()

print("✅ 模型加载成功！")

# ===================== 预处理 =====================
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ===================== 特征提取 =====================
def extract_feature(img_np):
    with torch.no_grad():
        img = preprocess(img_np).unsqueeze(0).to(DEVICE)
        feat = model(img)
        feat = feat.squeeze(0).cpu().numpy()
    return feat

# ===================== 批量提取 =====================
def extract_all():
    files = [f for f in os.listdir(NPZ_DIR) if f.endswith(".npz")]
    print(f"📊 共 {len(files)} 只股票")

    for f in tqdm(files, desc="提取图像特征"):
        sid = f[:-4]
        data = np.load(os.path.join(NPZ_DIR, f))
        
        imgs = data["cv"]
        labels = data["label"]
        
        feats = [extract_feature(img) for img in imgs]
        np.savez_compressed(
            os.path.join(SAVE_FEATURE_DIR, f"{sid}_mae.npz"),
            feature=np.array(feats),
            label=labels
        )

    print("🎉 全部完成！图像特征已准备好用于多模态训练！")

if __name__ == "__main__":
    extract_all()

🚀 加载本地视觉模型...
✅ 模型加载成功！
📊 共 270 只股票


提取图像特征: 100%|██████████| 270/270 [19:02<00:00,  4.23s/it]

🎉 全部完成！图像特征已准备好用于多模态训练！
